# 手写数字识别

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as transforms
import torchvision.datasets as datasets

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [4]:
train_dataset = datasets.MNIST(
    root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(
    root='./data', train=False, download=True, transform=transform)

In [5]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [6]:
class mnistModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")  # Set the device to CUDA, MPS, or CPU
print(f"Using device: {device}")
model = mnistModel().to(device)  # Move the model to the selected device (GPU, MPS, or CPU)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Using device: mps


In [8]:
def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train()
    running_loss = 0.0
    #correct是当前批次的正确预测的样本数，total是总样本数
    correct = 0
    total = 0
#这里的enumerate(dataloader)会返回两个值：batch_idx是当前批次的索引，(data, target)是当前批次的数据和标签。
    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)  # Move the data and target to the selected device

        optimizer.zero_grad()  # Clear the gradients
        output = model(data)  # Forward pass
        loss = loss_fn(output, target)  # Compute the loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update the model parameters

        # track the running loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(output.data, 1)  # Get the index of the max log-probability
        total += target.size(0)  # Update the total number of samples
        correct += (predicted == target).sum().item()  # Update the

        # print the loss and accuracy every 100 batches
        if batch_idx % 100 == 99:
            print(f'Batch {batch_idx + 1}, Loss: {running_loss / 100:.4f}, Accuracy: {100 * correct / total:.2f}%')
            running_loss = 0.0
            correct = 0
            total = 0

In [9]:
def evaluate(model, test_dataset, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_dataset:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    return 100 * correct / total

In [10]:
num_epochs = 5

for epoch in range(num_epochs):
    print(f'Epoch {epoch + 1}/{num_epochs}')
    train_epoch(model, train_loader, loss_fn, optimizer, device)
    accuracy = evaluate(model, test_loader, device)
    print(f'Test Accuracy: {accuracy:.2f}%\n')

Epoch 1/5
Batch 100, Loss: 0.7634, Accuracy: 76.91%
Batch 200, Loss: 0.3269, Accuracy: 90.62%
Batch 300, Loss: 0.2632, Accuracy: 92.02%
Batch 400, Loss: 0.2323, Accuracy: 93.06%
Batch 500, Loss: 0.1990, Accuracy: 94.27%
Batch 600, Loss: 0.1904, Accuracy: 94.09%
Batch 700, Loss: 0.1595, Accuracy: 95.33%
Batch 800, Loss: 0.1583, Accuracy: 95.20%
Batch 900, Loss: 0.1554, Accuracy: 95.38%
Test Accuracy: 95.83%

Epoch 2/5
Batch 100, Loss: 0.1269, Accuracy: 96.22%
Batch 200, Loss: 0.1060, Accuracy: 96.62%
Batch 300, Loss: 0.1215, Accuracy: 96.34%
Batch 400, Loss: 0.1103, Accuracy: 96.62%
Batch 500, Loss: 0.1049, Accuracy: 96.72%
Batch 600, Loss: 0.1122, Accuracy: 96.61%
Batch 700, Loss: 0.1064, Accuracy: 96.42%
Batch 800, Loss: 0.0956, Accuracy: 96.98%
Batch 900, Loss: 0.1128, Accuracy: 96.80%
Test Accuracy: 96.93%

Epoch 3/5
Batch 100, Loss: 0.0752, Accuracy: 97.67%
Batch 200, Loss: 0.0727, Accuracy: 97.67%
Batch 300, Loss: 0.0769, Accuracy: 97.58%
Batch 400, Loss: 0.0794, Accuracy: 97.67%
